In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

from src.paths import PROCESSED_PATH, RAW_PATH
from src.portfolio import classify_economic_type, is_likely_dca_asset

## 1. Load and analyze data from trade republic

In [ ]:
drop_cols = [
    "payment_reference",
    "counterparty_name",
    "counterparty_iban"
]

df_tran = pd.read_csv(RAW_PATH / "data.csv", sep=",")
df_tran = df_tran.drop(
    columns=drop_cols
)
df_tran.head()

In [ ]:
df_tran["date"] = pd.to_datetime(df_tran["date"])
df_tran = df_tran.sort_values("date")
df_tran.head()

In [ ]:
df_tran.columns.to_list()

In [ ]:
df_tran.info()

In [ ]:
df_tran["type"].value_counts()

In [ ]:
df_tran["category"].value_counts()

In [ ]:
df_tran["asset_class"].value_counts()

In [ ]:
df_tran["account_type"].value_counts()

In [ ]:
df_tran["name"].value_counts()

In [ ]:
df_tran["symbol"].value_counts()

In [ ]:
df_tran.groupby(
    ["category","type"]
).size()

In [ ]:
df_tran.notna().sum()

## 2. Categorization

In [ ]:
df_tran[
    df_tran["type"]=="EARNINGS"
][[
    "date",
    "amount",
    "description",
    "name"
]]

In [ ]:
df_tran[
df_tran["type"]
=="TRANSFER_DIRECT_DEBIT_INBOUND"
][[
"date",
"amount",
"description"
]]

In [ ]:
df_tran[
df_tran["type"]
=="CUSTOMER_INPAYMENT"
][[
"date",
"amount",
"description"
]]

In [ ]:
df_tran[
df_tran["type"]
=="CUSTOMER_OUTBOUND_REQUEST"
][[
"date",
"amount",
"description"
]]

In [ ]:
df_tran[
df_tran["type"]
=="EXCHANGE"
]

In [ ]:
df_tran[
df_tran["type"]
=="BENEFITS_SAVEBACK"
]

In [ ]:
df_tran["economic_type"] = (
    df_tran["type"]
    .map(classify_economic_type)
)

In [ ]:
df_tran["economic_type"].value_counts()

In [ ]:
df_tran[df_tran["economic_type"].isna()]["type"].value_counts()

In [ ]:
df_tran.groupby("economic_type")["amount"].sum().sort_values()


## 3. Cash Flows

### 3.1 Cashflow and consumption analysis

In [ ]:
df_cash = df_tran[
    df_tran["category"] == "CASH"
].copy()
df_cash.head()

In [ ]:
df_cash["month"] = (
    df_cash["date"]
    .dt.to_period("M")
)

monthly_cash = (
    df_cash
    .groupby("month")["amount"]
    .sum()
)

monthly_cash.plot(
    figsize=(12,5),
    title="Net Monthly Cash Flow"
)

In [ ]:
monthly_type_all = (
    df_cash
    .groupby(["month", "economic_type"])["amount"]
    .sum()
    .unstack(fill_value=0)
)

monthly_type_all.plot(
    figsize=(14, 6),
    title="Monthly Cash Flows by Type"
)

plt.xlabel("Month")
plt.ylabel("Amount")
plt.tight_layout()
plt.show()

In [ ]:
# Start of the cash-flow analysis. None = from the first card payment or direct
# debit, i.e. once the account is used for everyday spending (before that, cash
# flows are typically incomplete). Or set a date, e.g. "2024-01-01".
CASH_ANALYSIS_START = None

if CASH_ANALYSIS_START is None:
    CASH_ANALYSIS_START = df_cash.loc[df_cash["economic_type"] == "Consumption", "date"].min()
CASH_ANALYSIS_START = pd.Timestamp(CASH_ANALYSIS_START)

df_cash_recent = df_cash[
    df_cash["date"] >= CASH_ANALYSIS_START
].copy()

In [ ]:
monthly_cash_recent = (
    df_cash_recent
    .groupby("month")
    ["amount"]
    .sum()
)

monthly_cash_recent.plot(
    figsize=(12,5),
    title=f"Net Monthly Cash Flow (since {CASH_ANALYSIS_START:%b %Y})"
)

In [ ]:
monthly_type_recent = (
    df_cash_recent
    .groupby(["month", "economic_type"])["amount"]
    .sum()
    .unstack(fill_value=0)
)

monthly_type_recent.plot(
    figsize=(14, 6),
    title=f"Monthly Cash Flows by Type (since {CASH_ANALYSIS_START:%b %Y})"
)

plt.xlabel("Month")
plt.ylabel("Amount")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    monthly_type_recent.index.to_timestamp(),
    monthly_type_recent["Cash Inflow"],
    label="Income",
    marker="o"
)

plt.title(f"Monthly Cash Inflows (since {CASH_ANALYSIS_START:%b %Y})")
plt.xlabel("Date")
plt.ylabel("Amount")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    monthly_type_recent.index.to_timestamp(),
    -monthly_type_recent["Consumption"],
    label="Consumption",
    marker="s"
)

plt.title(f"Monthly Consumption (since {CASH_ANALYSIS_START:%b %Y})")
plt.xlabel("Date")
plt.ylabel("Amount")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_cash_sorted = df_cash.sort_values("date").copy()

cumulative_cash_flow = df_cash_sorted["amount"].cumsum()

plt.figure(figsize=(12,5))
plt.plot(
    df_cash_sorted["date"],
    cumulative_cash_flow
)

plt.title("Cumulative Cash Flow (CASH category only)")
plt.ylabel("€")
plt.tight_layout()
plt.show()

In [ ]:
rolling_cash = (
    monthly_cash_recent
    .rolling(3)
    .mean()
)

rolling_cash.plot(
    figsize=(12,5),
    title="3-Month Average Cash Flow"
)

In [ ]:
cash_summary = pd.DataFrame({
    "mean":[monthly_cash_recent.mean()],
    "median":[monthly_cash_recent.median()],
    "std":[monthly_cash_recent.std()],
    "min":[monthly_cash_recent.min()],
    "max":[monthly_cash_recent.max()]
})

cash_summary

In [ ]:
monthly_cash_recent

In [ ]:
card = df_cash_recent[
    df_cash_recent["economic_type"]=="Consumption"
]

monthly_card = (
    card
    .groupby("month")
    ["amount"]
    .sum()
)

monthly_card.describe()

In [ ]:
income = df_cash_recent[
    df_cash_recent["economic_type"]=="Cash Inflow"
]

monthly_income = (
    income
    .groupby("month")
    ["amount"]
    .sum()
)

monthly_income.describe()

#### 3.2 Passive income analysis

In [ ]:
passive = df_cash[
    df_cash["type"]
    .isin(
        [
            "DIVIDEND",
            "INTEREST_PAYMENT"
        ]
    )
]

monthly_passive = (
    passive
    .groupby("month")
    ["amount"]
    .sum()
)

monthly_passive.plot(
    figsize=(12,5),
    title="Monthly Passive Income"
)

plt.ylabel("€")
plt.xlabel("Date")
plt.show()

In [ ]:
monthly_passive.describe()

## 4. Investment Analysis

### 4.1 Prepare Dataset

In [ ]:
trading = df_tran[
    df_tran["category"] == "TRADING"
].copy()

trading["month"] = trading["date"].dt.to_period("M")

buys = trading[
    trading["type"] == "BUY"
].copy()

buys["buy_amount"] = buys["amount"].abs()

sells = trading[
    trading["type"] == "SELL"
].copy()

In [ ]:
trading.head()

### 4.2 Monthly investment volume

In [ ]:
monthly_buy_volume = (
    buys
    .groupby("month")["buy_amount"]
    .sum()
)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(
    monthly_buy_volume.index.to_timestamp(),
    monthly_buy_volume.values,
    label="Buy Volume"
)
plt.title("Monthly Buy Volume")
plt.ylabel("€")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.show()

### 4.3 Buys per month

In [ ]:
monthly_buy_count = (
    buys
    .groupby("month")
    .size()
)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(
    monthly_buy_count.index.to_timestamp(),
    monthly_buy_count.values,
    label="Buy Count",
    color="orange"
)
plt.title("Monthly Buy Count")
plt.ylabel("Number of Buys")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.show()

### 4.4 Average order size per month

In [ ]:
avg_order_size = (
    monthly_buy_volume / monthly_buy_count
)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(
    avg_order_size.index.to_timestamp(),
    avg_order_size.values,
    label="Average Order Size",
    color="blue"
)
plt.title("Monthly Average Order Size")
plt.ylabel("Average Order Size (€) ")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.show()

### 4.5 Asset class distribution

In [ ]:
buy_volume_by_asset_class = (
    buys
    .groupby("asset_class")["buy_amount"]
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
plt.figure(figsize=(12,5))
plt.bar(
    buy_volume_by_asset_class.index,
    buy_volume_by_asset_class.values
)
plt.title("Buy Volume by Asset Class")
plt.ylabel("€")
plt.xlabel("Asset Class")
plt.tight_layout()
plt.show()

### 4.6 Symbol distribution

In [ ]:
buy_volume_by_asset = (
    buys
    .groupby(["symbol", "name"])["buy_amount"]
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
plt.figure(figsize=(12,5))
plt.barh(
    buy_volume_by_asset.index.get_level_values(1),
    buy_volume_by_asset.values
)
plt.title("Buy Volume by Name")
plt.ylabel("Name")
plt.xlabel("€")
plt.tight_layout()
plt.show()

### 4.7 Fees and taxes

In [ ]:
total_fees = trading["fee"].sum()
total_taxes = df_tran["tax"].sum()

In [ ]:
total_fees

In [ ]:
total_taxes

In [ ]:
fees_by_year = (
    trading
    .groupby(trading["date"].dt.year)["fee"]
    .sum()
)
fees_by_year

In [ ]:
taxes_by_year = (
    df_tran
    .groupby(df_tran["date"].dt.year)["tax"]
    .sum()
)
taxes_by_year

### 4.8 Sell Analysis

In [ ]:
sells[[
    "date",
    "name",
    "symbol",
    "shares",
    "price",
    "amount",
    "fee",
    "tax"
]]

### 4.9 DCA / Savings Plan Behaviour

In [ ]:
monthly_buys_by_asset = (
    buys
    .groupby(["month", "symbol"])["buy_amount"]
    .sum()
    .unstack(fill_value=0)
)
monthly_buys_by_asset

In [ ]:
top_dca_assets = (
    buys
    .groupby("name")["buy_amount"]
    .sum()
    .sort_values(ascending=False)
    .head(8)
)

top_dca_assets

In [ ]:
plt.figure(figsize=(12,5))
plt.barh(
    top_dca_assets.index,
    top_dca_assets.values
)
plt.title("Top DCA Assets by Total Investment")
plt.ylabel("Name")
plt.xlabel("Total Investment (€)")
plt.tight_layout()
plt.show()

In [ ]:
dca_summary = []

for symbol, data in buys.groupby("symbol"):
    
    monthly = (
        data
        .groupby("month")["buy_amount"]
        .sum()
    )
    
    full_months = pd.period_range(
        start=monthly.index.min(),
        end=monthly.index.max(),
        freq="M"
    )
    
    monthly_full = monthly.reindex(full_months, fill_value=0)
    active_monthly = monthly_full[monthly_full > 0]
    
    active_months = len(active_monthly)
    total_months = len(monthly_full)
    
    dca_summary.append({
        "symbol": symbol,
        "name": data["name"].iloc[0],
        "first_month": monthly.index.min(),
        "last_month": monthly.index.max(),
        "total_invested": monthly.sum(),
        "n_orders": len(data),
        "active_months": active_months,
        "total_months": total_months,
        "regularity_rate": active_months / total_months,
        "avg_active_monthly_investment": active_monthly.mean(),
        "std_active_monthly_investment": active_monthly.std(ddof=0),
        "avg_order_size": data["buy_amount"].mean()
    })

dca_summary = pd.DataFrame(dca_summary)

dca_summary = dca_summary.sort_values(
    "total_invested",
    ascending=False
)

dca_summary

In [ ]:
dca_summary["cv_active_monthly_investment"] = (
    dca_summary["std_active_monthly_investment"]
    / dca_summary["avg_active_monthly_investment"]
)

dca_summary.sort_values("regularity_rate", ascending=False)

In [ ]:
dca_summary["likely_dca_asset"] = is_likely_dca_asset(
    dca_summary["n_orders"],
    dca_summary["active_months"],
    dca_summary["total_months"],
    dca_summary["regularity_rate"]
)

dca_summary[
    [
        "symbol",
        "name",
        "total_invested",
        "n_orders",
        "active_months",
        "total_months",
        "regularity_rate",
        "avg_order_size",
        "cv_active_monthly_investment",
        "likely_dca_asset"
    ]
].sort_values("total_invested", ascending=False)

In [ ]:
likely_dca_symbols = (
    dca_summary
    .loc[dca_summary["likely_dca_asset"], "symbol"]
    .tolist()
)

monthly_buys_by_asset[likely_dca_symbols].plot(
    figsize=(14,6),
    marker="o",
    title="Likely DCA Assets: Monthly Investment Pattern"
)

plt.xlabel("Month")
plt.ylabel("Investment Amount (€)")
plt.tight_layout()
plt.show()

## 5. Corporate Actions

In [ ]:
corporate_actions = df_tran[
    df_tran["economic_type"] == "Corporate Action"
]
corporate_actions

## 6. Save data and KPIs

In [ ]:
df_tran.to_csv(PROCESSED_PATH / "transactions_clean.csv", index=False)

In [ ]:
dashboard_kpis = pd.DataFrame([{
    "n_transactions": len(df_tran),
    "n_cash_transactions": (df_tran["category"] == "CASH").sum(),
    "n_trading_transactions": (df_tran["category"] == "TRADING").sum(),
    "n_corporate_actions": (df_tran["category"] == "CORPORATE_ACTION").sum(),

    "total_buy_volume": buys["buy_amount"].sum(),
    "total_sell_volume": sells["amount"].sum(),
    "net_invested": buys["buy_amount"].sum() - sells["amount"].sum(),

    "n_buy_orders": len(buys),
    "n_sell_orders": len(sells),
    "avg_buy_order_size": buys["buy_amount"].mean(),

    "fees_paid": -trading["fee"].sum(),
    "taxes_paid": -df_tran["tax"].sum(),

    "monthly_cashflow_mean_recent": monthly_cash_recent.mean(),
    "monthly_cashflow_median_recent": monthly_cash_recent.median(),
    "monthly_cashflow_std_recent": monthly_cash_recent.std(),

    "monthly_consumption_mean_recent": -monthly_card.mean(),
    "monthly_income_mean_recent": monthly_income.mean(),
    "passive_income_total": monthly_passive.sum()
}])

dashboard_kpis.to_csv(
    PROCESSED_PATH / "transaction_dashboard_kpis.csv",
    index=False
)

dashboard_kpis

In [ ]:
_current_month = pd.Timestamp.now().to_period("M")

recent_months = pd.period_range(
    start=monthly_cash_recent.index.min(),
    end=min(monthly_cash_recent.index.max(), _current_month - 1),
    freq="M"
)

monthly_dashboard = pd.DataFrame(index=recent_months)

monthly_dashboard["net_cash_flow"] = monthly_cash_recent.reindex(recent_months, fill_value=0)
monthly_dashboard["consumption"] = -monthly_card.reindex(recent_months, fill_value=0)
monthly_dashboard["cash_inflow"] = monthly_income.reindex(recent_months, fill_value=0)
monthly_dashboard["buy_volume"] = monthly_buy_volume.reindex(recent_months, fill_value=0)

monthly_dashboard = monthly_dashboard.reset_index().rename(columns={"index": "month"})
monthly_dashboard["month"] = monthly_dashboard["month"].astype(str)

monthly_dashboard.to_csv(
    PROCESSED_PATH / "monthly_transaction_dashboard.csv",
    index=False
)

monthly_dashboard

In [ ]:
dca_summary.to_csv(
    PROCESSED_PATH / "dca_summary.csv",
    index=False
)